In [13]:
import requests
import pandas as pd
import time

In [14]:
resp = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios")
resp.raise_for_status()
municipios = resp.json()

df_municipios = pd.json_normalize(municipios)
df_municipios = df_municipios[[
    "id", "nome",
    "microrregiao.mesorregiao.UF.sigla",
    "microrregiao.mesorregiao.UF.nome",
]].rename(columns={
    "id": "codigo_ibge",
    "nome": "nome_municipio",
    "microrregiao.mesorregiao.UF.sigla": "uf",
    "microrregiao.mesorregiao.UF.nome": "estado",
})

print(f"{len(df_municipios)} municípios encontrados")

5571 municípios encontrados


In [15]:
#  Descobre quais anos existem de fato pra cada tabela, a partir de 2010
def obter_periodos_validos(agregado, ano_inicio=2010):
    url = f"https://servicodados.ibge.gov.br/api/v3/agregados/{agregado}/periodos"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    periodos = resp.json()
    anos = sorted(int(p["id"]) for p in periodos if p["id"].isdigit())
    return [a for a in anos if a >= ano_inicio]

In [16]:
# Função genérica pra buscar uma variável do SIDRA, em vários anos e em lote de municípios ----------

def buscar_variavel_sidra(agregado, variavel, codigos, anos, tamanho_lote=200, pausa=0.3):
    periodo_str = ",".join(str(a) for a in anos)  # ex: "2010,2011,2012,...,2024"
    linhas = []
    total_lotes = (len(codigos) - 1) // tamanho_lote + 1

    for i in range(0, len(codigos), tamanho_lote):
        lote = codigos[i:i + tamanho_lote]
        codigos_str = ",".join(str(int(c)) for c in lote)

        url = f"https://servicodados.ibge.gov.br/api/v3/agregados/{agregado}/periodos/{periodo_str}/variaveis/{variavel}"
        resp = requests.get(url, params={"localidades": f"N6[{codigos_str}]"}, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        for serie_info in data[0]["resultados"][0]["series"]:
            cod = int(serie_info["localidade"]["id"])
            for ano, valor in serie_info["serie"].items():
                linhas.append({"codigo_ibge": cod, "ano": int(ano), "valor": valor})

        print(f"  lote {i // tamanho_lote + 1}/{total_lotes} ok")
        time.sleep(pausa)

    return pd.DataFrame(linhas)

todos_codigos = df_municipios["codigo_ibge"].tolist()

In [26]:
anos_populacao

[2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2024,
 2025,
 2026]

In [17]:
# População (tabela 6579, variável 9324), desde 2010 ----------

anos_populacao = obter_periodos_validos(agregado=6579, ano_inicio=2010)
print(f"Anos disponíveis de população: {anos_populacao[0]}–{anos_populacao[-1]}")

populacao = buscar_variavel_sidra(agregado=6579, variavel=9324, codigos=todos_codigos, anos=anos_populacao)
populacao["valor"] = pd.to_numeric(populacao["valor"], errors="coerce")
populacao = populacao.rename(columns={"valor": "populacao"})

Anos disponíveis de população: 2011–2026
  lote 1/28 ok
  lote 2/28 ok
  lote 3/28 ok
  lote 4/28 ok
  lote 5/28 ok
  lote 6/28 ok
  lote 7/28 ok
  lote 8/28 ok
  lote 9/28 ok
  lote 10/28 ok
  lote 11/28 ok
  lote 12/28 ok
  lote 13/28 ok
  lote 14/28 ok
  lote 15/28 ok
  lote 16/28 ok
  lote 17/28 ok
  lote 18/28 ok
  lote 19/28 ok
  lote 20/28 ok
  lote 21/28 ok
  lote 22/28 ok
  lote 23/28 ok
  lote 24/28 ok
  lote 25/28 ok
  lote 26/28 ok
  lote 27/28 ok
  lote 28/28 ok


In [18]:
#  PIB municipal (tabela 5938, variável 37), desde 2010 ----------

anos_pib = obter_periodos_validos(agregado=5938, ano_inicio=2010)
print(f"Anos disponíveis de PIB: {anos_pib[0]}–{anos_pib[-1]}")

pib = buscar_variavel_sidra(agregado=5938, variavel=37, codigos=todos_codigos, anos=anos_pib)
pib["valor"] = pd.to_numeric(pib["valor"], errors="coerce")
pib = pib.rename(columns={"valor": "pib"})

Anos disponíveis de PIB: 2010–2023
  lote 1/28 ok
  lote 2/28 ok
  lote 3/28 ok
  lote 4/28 ok
  lote 5/28 ok
  lote 6/28 ok
  lote 7/28 ok
  lote 8/28 ok
  lote 9/28 ok
  lote 10/28 ok
  lote 11/28 ok
  lote 12/28 ok
  lote 13/28 ok
  lote 14/28 ok
  lote 15/28 ok
  lote 16/28 ok
  lote 17/28 ok
  lote 18/28 ok
  lote 19/28 ok
  lote 20/28 ok
  lote 21/28 ok
  lote 22/28 ok
  lote 23/28 ok
  lote 24/28 ok
  lote 25/28 ok
  lote 26/28 ok
  lote 27/28 ok
  lote 28/28 ok


In [19]:
# Junta tudo: um tabelão único, uma linha por (município, ano) ----------

serie_historica = populacao.merge(pib, on=["codigo_ibge", "ano"], how="outer")
serie_historica = df_municipios.merge(serie_historica, on="codigo_ibge", how="left")
serie_historica = serie_historica.sort_values(["codigo_ibge", "ano"]).reset_index(drop=True)

serie_historica.to_csv("municipios_brasil_serie_historica_2010.csv", index=False)
print(serie_historica.shape)


(94707, 7)


In [20]:
serie_historica.head(20)

,codigo_ibge,nome_municipio,uf,estado,ano,populacao,pib
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,2010,NaN,262077.0
1,1100015,Alta Floresta D'Oeste,RO,Rondônia,2011,24228.0,280510.0
2,1100015,Alta Floresta D'Oeste,RO,Rondônia,2012,24069.0,329029.0
3,1100015,Alta Floresta D'Oeste,RO,Rondônia,2013,25728.0,341325.0
4,1100015,Alta Floresta D'Oeste,RO,Rondônia,2014,25652.0,377799.0
5,1100015,Alta Floresta D'Oeste,RO,Rondônia,2015,25578.0,421300.0
6,1100015,Alta Floresta D'Oeste,RO,Rondônia,2016,25506.0,478217.0
7,1100015,Alta Floresta D'Oeste,RO,Rondônia,2017,25437.0,485374.0
8,1100015,Alta Floresta D'Oeste,RO,Rondônia,2018,23167.0,498980.0
9,1100015,Alta Floresta D'Oeste,RO,Rondônia,2019,22945.0,495775.0


In [ ]:
serie_historica.to_csv()

In [ ]:
# O Censo 2022 foi seriamente atrasado. Ele estava previsto pra 2020, mas foi adiado pela pandemia. Em 2021 o governo cortou mais de 90% do orçamento do Censo, quase inviabilizando a operação (o IBGE chegou a dizer que o planejamento já tinha sido desmontado). O STF só liberou fazer o Censo em 2022, e a coleta de dados começou de fato em agosto de 2022 — com o prazo de coleta tendo que ser estendido até dezembro daquele ano porque em outubro só 49% da população tinha sido contada. Os resultados oficiais completos só saíram em 28 de junho de 2023.

# O PIB dos Municípios tem uma defasagem estrutural de cerca de 2 anos entre o ano de referência e a divulgação. O dado de 2023, por exemplo, só foi divulgado em 19 de dezembro de 2025. Isso acontece porque, diferente do PIB nacional (que sai trimestralmente e rápido), o PIB municipal é calculado cruzando várias fontes diferentes — dados da Receita Federal, das empresas, da produção agropecuária, de pesquisas industriais, contas regionais etc. — e reconciliar tudo isso no nível de cada um dos 5.570 municípios leva tempo.
# Então, hoje (agosto de 2026), o último ano disponível pra PIB municipal ainda é 2023. Seguindo esse padrão de ~2 anos de atraso, o dado de 2024 deve sair só no final de 2026 ou já em 2027 — e 2025/2026 nem existem ainda como possibilidade, porque os anos ainda não terminaram nem foram fechados contabilmente.

# Um detalhe a mais: o próprio IBGE informou que a divulgação do PIB municipal detalhado por setor de atividade (indústria, agropecuária, serviços) ficou temporariamente suspensa e só volta em 2027, por causa da atualização da série de Contas Nacionais pra uma nova base (ano-base 2021) — ou seja, teve até uma pausa metodológica extra nessa granularidade, embora o valor total do PIB continue sendo publicado normalmente.